To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os, zipfile
from google.colab import files

output_dir = "/content/faseF_v2_checkpoints"
os.makedirs(output_dir, exist_ok=True)

'''
subido = files.upload()
nombre_zip = list(subido.keys())[0]
with zipfile.ZipFile(nombre_zip, 'r') as zip_ref:
  zip_ref.extractall(output_dir)
'''

checkpoints = [f for f in os.listdir(output_dir) if f.startswith("checkpoint-")]
ya_existe_checkpoint = len(checkpoints) > 0
step_actual = max(int(c.split("-")[1]) for c in checkpoints) if ya_existe_checkpoint else 0

print(f"Step actual (tramo 2): {step_actual}")
print(f"Ya existe checkpoint del tramo 2: {ya_existe_checkpoint}")

Step actual (tramo 2): 0
Ya existe checkpoint del tramo 2: False


### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0.05,
    bias = "none",
    random_state = 3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
# Ejecutar SOLO si es la 1a sesion del tramo 2 y el checkpoint-1200 no esta ya en el disco
print("Sube el zip que contiene checkpoint-400 (del tramo 1):")
subido_origen = files.upload()
nombre_zip_origen = list(subido_origen.keys())[0]
with zipfile.ZipFile(nombre_zip_origen, 'r') as zip_ref:
    zip_ref.extractall("/content/faseF_checkpoints")

Sube el zip que contiene checkpoint-400 (del tramo 1):


Saving checkpoint_step_400.zip to checkpoint_step_400.zip


In [ ]:
from safetensors.torch import load_file

if not ya_existe_checkpoint:
    ruta_checkpoint_origen = "/content/faseF_checkpoints/checkpoint-400/adapter_model.safetensors"
    pesos_checkpoint = load_file(ruta_checkpoint_origen)
    model.load_state_dict(pesos_checkpoint, strict=False)
    print("Pesos del checkpoint-600 (tramo 1) cargados como punto de partida")
else:
    print("Tramo 2 ya iniciado: se omite la carga del checkpoint-1200 original")

Pesos del checkpoint-600 (tramo 1) cargados como punto de partida


In [ ]:
from datasets import load_dataset, concatenate_datasets

dataset_alcazaba_completo = load_dataset("csv", data_files="/content/sample_data/dataset_fortaleza_roja_200.txt", split="train")
dataset_jardin = load_dataset("csv", data_files="/content/sample_data/dataset_palacios_sigloXIII.txt", split="train")

PORCENTAJE_REPASO = 1.0
n_repaso = int(len(dataset_alcazaba_completo) * PORCENTAJE_REPASO)
dataset_alcazaba_repaso = dataset_alcazaba_completo.shuffle(seed=3407).select(range(n_repaso))

dataset = concatenate_datasets([dataset_alcazaba_repaso, dataset_jardin])
dataset = dataset.shuffle(seed=3407)

#dataset = dataset_alcazaba_completo
print(f"Dataset del tramo 2: {len(dataset)} preguntas totales")
print(f"  - Repaso Alcazaba: {len(dataset_alcazaba_repaso)}")
#print(f"  - Jardin Feliz (nuevo): {len(dataset_jardin)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset del tramo 2: 400 preguntas totales
  - Repaso Alcazaba: 200


We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [ ]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": "Eres un guia experto en la Alhambra. "
             "Capaz de responder preguntas de la historia y de medidas de los "
             "edificios correspondientes. Ademas de ser honesto pues si no "
             "sabes la respuesta indicas que no la sabes."},
            {"role": "user", "content": example["pregunta"]},
            {"role": "assistant", "content": example["respuesta"]}
        ]
    }
dataset = dataset.map(convert_to_chatml)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False,
             add_generation_prompt=False).removeprefix('<bos>') for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Let's see how row 100 looks like!

In [ ]:
dataset[100]

{'id': 157,
 'pregunta': '¿Qué es el camino de ronda en la Alhambra?',
 'respuesta': 'El camino de ronda es el paseo sobre las murallas de la Alhambra desde el que los soldados vigilaban el exterior y podían moverse entre las torres. En el Palacio de Abencerrajes del siglo XIII, el camino de ronda discurría bajo la qubba de la torre palatina.',
 'conversations': [{'role': 'system',
   'content': 'Eres un guia experto en la Alhambra. Capaz de responder preguntas de la historia y de medidas de los edificios correspondientes. Ademas de ser honesto pues si no sabes la respuesta indicas que no la sabes.'},
  {'role': 'user', 'content': '¿Qué es el camino de ronda en la Alhambra?'},
  {'role': 'assistant',
   'content': 'El camino de ronda es el paseo sobre las murallas de la Alhambra desde el que los soldados vigilaban el exterior y podían moverse entre las torres. En el Palacio de Abencerrajes del siglo XIII, el camino de ronda discurría bajo la qubba de la torre palatina.'}],
 'text': '<s

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [ ]:
dataset[100]["text"]

'<start_of_turn>user\nEres un guia experto en la Alhambra. Capaz de responder preguntas de la historia y de medidas de los edificios correspondientes. Ademas de ser honesto pues si no sabes la respuesta indicas que no la sabes.\n\n¿Qué es el camino de ronda en la Alhambra?<end_of_turn>\n<start_of_turn>model\nEl camino de ronda es el paseo sobre las murallas de la Alhambra desde el que los soldados vigilaban el exterior y podían moverse entre las torres. En el Palacio de Abencerrajes del siglo XIII, el camino de ronda discurría bajo la qubba de la torre palatina.<end_of_turn>\n'

In [ ]:
NUM_EJEMPLOS_TRAMO2 = len(dataset)
EPOCAS_OBJETIVO = 8
BATCH_EFECTIVO = 4

OBJETIVO_FINAL = int((EPOCAS_OBJETIVO * NUM_EJEMPLOS_TRAMO2) / BATCH_EFECTIVO)
WARMUP_STEPS = int(OBJETIVO_FINAL * 0.10)
INCREMENTO_POR_TANDA = int(OBJETIVO_FINAL)
objetivo_esta_tanda = min(step_actual + INCREMENTO_POR_TANDA, OBJETIVO_FINAL)

print(f"max_steps total del tramo 2: {OBJETIVO_FINAL}")
print(f"warmup_steps: {WARMUP_STEPS}")
print(f"Esta tanda entrenara hasta el step: {objetivo_esta_tanda} / {OBJETIVO_FINAL}")

max_steps total del tramo 2: 800
warmup_steps: 80
Esta tanda entrenara hasta el step: 800 / 800


In [ ]:
from transformers import TrainerCallback
import shutil

class DescargaAutomaticaCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        nombre_zip = f"/content/faseF_v2_step_{state.global_step}"
        shutil.make_archive(nombre_zip, 'zip', output_dir)
        print(f"Checkpoint en step {state.global_step} guardado. Descargando...")
        files.download(nombre_zip + ".zip")

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    data_collator = UnslothVisionDataCollator(
        model, tokenizer,
        train_on_responses_only = True,
        instruction_part = "<start_of_turn>user\n",
        response_part = "<start_of_turn>model\n",
    ),
    args = SFTConfig(
        output_dir = output_dir,
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = WARMUP_STEPS,
        max_steps = objetivo_esta_tanda,
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        learning_rate = 2e-4,
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
    callbacks = [DescargaAutomaticaCallback()],
)

Unsloth: Switching to float32 training since model cannot work with float16


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes! Unsloth now auto-detects the instruction and response parts from the tokenizer's chat template, so we don't need to pass `instruction_part` and `response_part` anymore. You can still pass them explicitly if you use a custom chat template.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<start_of_turn>user\n' and response_part = '<start_of_turn>model\n'


In [ ]:
print("OBJETIVO_FINAL:", OBJETIVO_FINAL, type(OBJETIVO_FINAL))
print("WARMUP_STEPS:", WARMUP_STEPS, type(WARMUP_STEPS))
print("objetivo_esta_tanda:", objetivo_esta_tanda, type(objetivo_esta_tanda))
print("NUM_EJEMPLOS_TRAMO2:", NUM_EJEMPLOS_TRAMO2, type(NUM_EJEMPLOS_TRAMO2))

OBJETIVO_FINAL: 800 <class 'int'>
WARMUP_STEPS: 80 <class 'int'>
objetivo_esta_tanda: 800 <class 'int'>
NUM_EJEMPLOS_TRAMO2: 400 <class 'int'>


In [ ]:
if ya_existe_checkpoint:
    print("Reanudando tramo 2 desde su propio checkpoint...")
    stats = trainer.train(resume_from_checkpoint=True)
else:
    print("Arrancando tramo 2 desde los pesos del checkpoint-1200...")
    stats = trainer.train()

print("Step final de esta tanda:", trainer.state.global_step, "/", OBJETIVO_FINAL)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 8 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)


Arrancando tramo 2 desde los pesos del checkpoint-1200...


Step,Training Loss
20,3.800600
40,2.215200
60,1.874000
80,1.808300
100,1.659300
120,1.325500
140,1.232200
160,1.287400
180,1.286700
200,1.153600


Checkpoint en step 200 guardado. Descargando...


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Checkpoint en step 800 guardado. Descargando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Step final de esta tanda: 800 / 800


In [ ]:
shutil.make_archive('faseF_v4_400_backup_1', 'zip', output_dir)
files.download('faseF_v4_400_backup_1.zip')
print("Guarda ese zip: lo subiras en la Celda 2 de la proxima sesion")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Guarda ese zip: lo subiras en la Celda 2 de la proxima sesion
